# AI Pathfinding Visualizer
A desktop app that visualizes 6 AI search algorithms - BFS, DFS, A*, Hill Climbing, Best-first, and Minimax on an interactive 30x20 grid. Draw walls, place start/end points, pick a case mode and watch the algorithm explore in real time. 

# Pathfinding 
Pathfinding is the computational problem of finding a route between two points.

# 1. Install libraries

In [1]:
# This installs pygame. Run once, then you're set.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pygame==2.6.1"])
print("Pygame installed!")

Pygame installed!


# 2. Window geometry & color palette
The window is 1170 × 718 px, split into four regions:
HEADER (72px) → toolbar | 
GRID (840×560px) → 30×20 cells of 28px each | 
PANEL (330px wide) → stats | 
FOOTER (86px) → legend & speed

Colors live in a single COLORS dict so RGB values aren't scattered through render code. Cell states map directly to keys: "empty", "wall", "start", "end", "visited", "best_path", "avg_path", "worst_path".

In [2]:
# Part 1: Constants — Grid Dimensions

ROWS = 20          # Number of rows in the grid
COLS = 30          # Number of columns in the grid
CELL = 28          # Each cell is 28x28 pixels

PANEL_WIDTH = 330  # Width of the right-side info panel
HEADER_HEIGHT = 72 # Height of the top toolbar
FOOTER_HEIGHT = 86 # Height of the bottom legend/controls bar

GRID_WIDTH  = COLS * CELL                        # 30 * 28 = 840 px
GRID_HEIGHT = ROWS * CELL                        # 20 * 28 = 560 px
WINDOW_WIDTH  = GRID_WIDTH + PANEL_WIDTH         # 840 + 330 = 1170 px
WINDOW_HEIGHT = HEADER_HEIGHT + GRID_HEIGHT + FOOTER_HEIGHT  # 72+560+86 = 718 px

print(f"Window size: {WINDOW_WIDTH} x {WINDOW_HEIGHT} px")
print(f"Grid: {ROWS} rows x {COLS} cols, each cell = {CELL}px")

Window size: 1170 x 718 px
Grid: 20 rows x 30 cols, each cell = 28px


In [3]:
# Part 2: Color Palette
# All colors are RGB tuples: (Red, Green, Blue) each 0-255

COLORS = {
    # Background & structural colors
    "bg":           (245, 241, 230),  # warm off-white background
    "bg_alt":       (233, 242, 239),  # slightly cooler variant
    "panel":        (250, 247, 240),  # right panel background
    "panel_soft":   (240, 235, 225),  # footer/side panel
    "card":         (255, 255, 252),  # stat cards (near white)
    "border":       (182, 172, 155),  # main borders
    "border_soft":  (210, 201, 187),  # subtle borders
    "text":         (28,  35,  44 ),  # near-black text
    "muted":        (96,  104, 112),  # grey labels
    "chip_text":    (67,  75,  83 ),  # button text

    # Cell state colors
    "empty":        (247, 245, 238),  # unvisited cell
    "wall":         (44,  44,  42 ),  # blocked cell (near black)
    "start":        (29,  158, 117),  # green — origin
    "end":          (216, 90,  48 ),  # orange — destination
    "visited":      (181, 212, 244),  # light blue — cells explored
    "best_path":    (239, 159, 39 ),  # gold — optimal path
    "avg_path":     (127, 119, 221),  # purple — average path
    "worst_path":   (226, 75,  74 ),  # red — worst path
}

# Per-mode colors (used for buttons and path drawing)
CASE_COLORS = {
    "Best Case":    COLORS["start"],       # green
    "Average Case": COLORS["avg_path"],    # purple
    "Worst Case":   COLORS["worst_path"],  # red
}

print("Color palette defined:", list(COLORS.keys()))

Color palette defined: ['bg', 'bg_alt', 'panel', 'panel_soft', 'card', 'border', 'border_soft', 'text', 'muted', 'chip_text', 'empty', 'wall', 'start', 'end', 'visited', 'best_path', 'avg_path', 'worst_path']


In [4]:
# Part 3: Algorithm metadata

# Human-readable descriptions shown in the side panel
ALGO_BLURB = {
    "BFS":           "Explores level-by-level and guarantees shortest path on unweighted grids.",
    "DFS":           "Dives deep into one branch first and backtracks when blocked.",
    "A*":            "Balances known cost and estimated distance for efficient optimal paths.",
    "Hill Climbing": "Greedy local search moving to better immediate neighbors.",
    "Best First":    "Expands the most promising node according to heuristic only.",
    "MiniMax":       "Adversarial-style exploration with alternating maximize/minimize choices.",
}

# Time complexity for each algorithm: (Best, Average, Worst)
# V = vertices, E = edges, b = branching factor, d = depth, m = max depth
TIME_COMPLEXITY = {
    "BFS":           ("O(1)",   "O(V+E)",   "O(V+E)"),
    "DFS":           ("O(1)",   "O(V+E)",   "O(V+E)"),
    "A*":            ("O(b^d)", "O(b^d)",   "O(b^d)"),
    "Hill Climbing": ("O(b·m)", "O(b·m)",   "O(b·m)"),
    "Best First":    ("O(b^m)", "O(b^m)",   "O(b^m)"),
    "MiniMax":       ("O(b^m)", "O(b^m)",   "O(b^m)"),
}

# Available algorithm names and mode names (used for UI button rendering)
ALGORITHMS = ["BFS", "DFS", "A*", "Hill Climbing", "Best First", "MiniMax"]
MODES      = ["Best Case", "Average Case", "Worst Case"]

# Tool constants — which tool is active on left-click
TOOL_WALL  = "wall"
TOOL_START = "start"
TOOL_END   = "end"

print("Algorithms:", ALGORITHMS)
print("Modes:", MODES)

Algorithms: ['BFS', 'DFS', 'A*', 'Hill Climbing', 'Best First', 'MiniMax']
Modes: ['Best Case', 'Average Case', 'Worst Case']


# 3. Grid Utilities
Three small but critical functions.

### Function :                      Purpose
- make_grid():                creates an empty 2D grid
- neighbors(grid,node):         returns all valid adjacent cells
- reconstruct_path(parent, end):  Traces back the shortest path (the parent dict fron end back to start and reverses)

In [5]:
# Grid utility functions

def make_grid():
    """
    Creates a 2D list (ROWS x COLS) where every cell starts as 'empty'.
    Each cell is a string: 'empty', 'wall', 'start', or 'end'.
    """
    return [["empty" for _ in range(COLS)] for _ in range(ROWS)]


def neighbors(grid, node):
    """
    Returns all valid (non-wall, in-bounds) cells adjacent to 'node'.
    We only move in 4 directions: up, down, left, right (no diagonals).

    node: (row, col) tuple
    Returns: list of (row, col) tuples

    The direction offsets (dr, dc) represent:
      (0, 1)  → move right
      (1, 0)  → move down
      (0, -1) → move left
      (-1, 0) → move up
    """
    r, c = node
    out = []
    for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        nr, nc = r + dr, c + dc
        # Boundary check: stay within grid
        # Wall check: skip blocked cells
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != "wall":
            out.append((nr, nc))
    return out


def reconstruct_path(parent, end):
    """
    Walks back through the 'parent' dictionary from 'end' to 'start'.
    parent[node] = the node we came FROM to reach this node.
    parent[start] = None  (the starting node has no predecessor).

    We collect the chain, then reverse it so path goes start→end.
    Returns [] if no valid path exists (only 1 node = just the start).
    """
    path = []
    curr = end
    while curr is not None:
        path.append(curr)
        curr = parent.get(curr)  # .get() returns None if key missing
    path.reverse()
    return path if len(path) > 1 else []  # empty if start==end or disconnected


# Quick sanity check:
test_grid = make_grid()
print(f"Grid shape: {len(test_grid)} rows x {len(test_grid[0])} cols")
print(f"Neighbors of (0,0):", neighbors(test_grid, (0, 0)))  # should be 2 (right + down)
print(f"Neighbors of (10,15):", neighbors(test_grid, (10, 15)))  # should be 4

Grid shape: 20 rows x 30 cols
Neighbors of (0,0): [(0, 1), (1, 0)]
Neighbors of (10,15): [(10, 16), (11, 15), (10, 14), (9, 15)]


# 5. The Heuristic Function
### What is heuristics?
    A heuristics is an extimate of how far  a cell, is from the goal. 
    It guids algorithms toward the destination without knowing the exact path.

### Manhattan Distance
We use the Manhattan distance- the number of steps needed if you can only move horizontally or vertically 
This is used by A*. Hill Climibing, Best-fit. and Minimax.


In [6]:
# The Heuristic Function

def heuristic(a, b):
    """
    Manhattan distance between two grid cells.
    a, b: (row, col) tuples
    Returns an integer >= 0.
    """
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


# Examples:
print(heuristic((0, 0), (0, 0)))    # 0  — same cell
print(heuristic((0, 0), (5, 7)))    # 12 — 5 down + 7 right
print(heuristic((10, 10), (0, 0)))  # 20 — 10 up + 10 left

0
12
20


# 5. Algorithm 1 - BFS(Breadth-first search)
BFS explores level by level, expanding all neighbors at distance 1 before at any distance 2.
It uses a queeu (FIFO- First In, First Out).
- Guarantees the shortest path 
on unweighed grid.
- Time and space: O(V+E)

In [7]:
# ---Code Cell----
from collections import deque  # deque = double-ended queue, O(1) popleft

def bfs(grid, start, end):
    """
    Breadth-First Search.

    Returns:
        visited: list of cells in the order they were explored (for animation)
        path:    the shortest path from start to end (empty if none found)
    """
    # Initialize: queue starts with only the start node
    q = deque([start])

    # 'seen' prevents revisiting the same cell
    seen = {start}

    # 'parent' maps each cell → where we came from
    # start came from nowhere, so its parent is None
    parent = {start: None}

    # We record every cell we visit for the animation
    visited = []

    while q:
        curr = q.popleft()         # Take from front of queue (FIFO)
        visited.append(curr)       # Record this cell as visited

        if curr == end:            # Found the destination!
            return visited, reconstruct_path(parent, end)

        for nb in neighbors(grid, curr):
            if nb not in seen:     # Only visit each cell once
                seen.add(nb)
                parent[nb] = curr  # Remember how we got here
                q.append(nb)       # Add to back of queue

    # If we exhaust the queue without finding end → no path
    return visited, []


# Test on a small grid:
g = make_grid()
v, p = bfs(g, (0,0), (3,4))
print(f"BFS visited {len(v)} cells, path length = {len(p)}")

BFS visited 32 cells, path length = 8


# 6. Algorithm 2 - DFS(Depth First Search)
### How DFS works:
DFS dives as deep as possible along one path before backtracking. It uses a stack (LIFO - Last In. First Out).
- Does not guarantee the shortest path
- Time: O(V+E) but explores in a very different order than BFS.

In [8]:
# --- CODE CELL ---
def dfs(grid, start, end):
    """
    Depth-First Search.
    Uses a Python list as a stack (append = push, pop = LIFO pop).
    """
    stack = [start]      # Start with only the start node
    seen = set()         # Track visited cells
    parent = {start: None}
    visited = []

    while stack:
        curr = stack.pop()   # Take from TOP of stack (LIFO — different from BFS!)

        if curr in seen:     # May push same node multiple times; skip if seen
            continue

        seen.add(curr)
        visited.append(curr)

        if curr == end:
            return visited, reconstruct_path(parent, end)

        for nb in neighbors(grid, curr):
            if nb not in seen:
                # Note: DFS updates parent here. Last write wins
                # because we may push the same neighbor from different paths.
                parent[nb] = curr
                stack.append(nb)   # Push to top of stack

    return visited, []


g = make_grid()
v, p = dfs(g, (0,0), (3,4))
print(f"DFS visited {len(v)} cells, path length = {len(p)}")

DFS visited 84 cells, path length = 84


# 7. Algorithm 3 - A*
### How A* Works
A* always finds the shortest path (if one exists) while exploring fae fewer cells than BFS.
It uses priority queue ordered by f(n) = g(n) +h(n) where:
        g(n) : actual cost from start to cell n
        h(n) : Heuristic (estimated cost from n to end)
        f(n) : Total estimated cost through n
A* always expands the cell with the lowest the cell with the lowest f score first.
This focuses the search toward the goal, unlike BFS which spreads uniformly.

In [9]:
# --- CODE CELL ---
def astar(grid, start, end):
    """
    A* Search.

    open_set entries: (f_score, g_score, node)
    We sort by f_score to always expand the most promising cell.

    g_score[n] = cost from start to n (in our unweighted grid, = steps)
    h(n)       = heuristic estimate from n to end
    f(n)       = g(n) + h(n)
    """
    # Initial f-score for start = 0 + heuristic(start, end)
    open_set = [(heuristic(start, end), 0, start)]
    seen = set()
    parent = {start: None}
    g_score = {start: 0}   # Actual cost from start to each node
    visited = []

    while open_set:
        open_set.sort(key=lambda x: x[0])  # Sort by f-score (lowest first)
        _, g_curr, curr = open_set.pop(0)   # Take lowest-f-score node

        if curr in seen:
            continue        # Skip stale entries (a better path was already found)

        seen.add(curr)
        visited.append(curr)

        if curr == end:
            return visited, reconstruct_path(parent, end)

        for nb in neighbors(grid, curr):
            # Each step costs 1 (unweighted grid)
            tentative_g = g_curr + 1

            # Only update if we found a CHEAPER path to nb
            if tentative_g < g_score.get(nb, float("inf")):
                g_score[nb] = tentative_g
                parent[nb] = curr
                h = heuristic(nb, end)
                f = tentative_g + h
                open_set.append((f, tentative_g, nb))

    return visited, []


g = make_grid()
v, p = astar(g, (0,0), (10,20))
print(f"A* visited {len(v)} cells, path length = {len(p)}")

A* visited 231 cells, path length = 31


# 8. Algorithm 4 - Hill Climbing 
### How Hill Climbing works
Hill Climbing is a greedy local search. At each step, it moves ro whichever neighbor is closest to the goal (by heuristic).
Current -> pick neighbor with lowest h(n) -> repeat
**Problem: Local maxima (or minima in our case)**
If all neighbors are farther from the goal than the current cell, Hill Climbing **stops**- it doesn't backtrack. This means it can fail even when a path exists.
This make it fast but **incomplete**- it won't always fins a path. 

In [10]:
# --- CODE CELL ---
def hill_climbing(grid, start, end):
    """
    Hill Climbing Search.
    Greedy — always moves to the neighbor closest to end.
    Stops if no neighbor is closer (gets stuck in local minimum).
    Does NOT use a queue; just follows one path.
    """
    curr = start
    seen = set()
    parent = {start: None}
    visited = []

    while curr is not None:
        if curr in seen:
            break           # Avoid cycles

        seen.add(curr)
        visited.append(curr)

        if curr == end:
            return visited, reconstruct_path(parent, end)

        # Get unseen neighbors and sort by heuristic (best = smallest h)
        nbs = [n for n in neighbors(grid, curr) if n not in seen]
        nbs.sort(key=lambda n: heuristic(n, end))

        # Move to best neighbor ONLY if it's closer than where we are
        if nbs and heuristic(nbs[0], end) < heuristic(curr, end):
            nxt = nbs[0]
            parent[nxt] = curr
            curr = nxt
        else:
            break  # Stuck! No better neighbor → stop

    return visited, []  # Failed to reach end


g = make_grid()
v, p = hill_climbing(g, (0,0), (10,20))
print(f"Hill Climbing visited {len(v)} cells, path = {len(p)} steps")

Hill Climbing visited 31 cells, path = 31 steps


# 9. Algorithm 5 - Best First Search
### How Best-First works
Best-first is like A* but **ignores the actual cost g(n)**. It only uses the heuristic h(n) to decide which cell to explore next. 
'''
Priority = h(n) only (compare: A* uses g(n)+h(n))
'''
This makes it faster than A* in practice (fewer calculations), but it's not optimal - it might find a longer path because it ignores how far it's already travelled.

In [11]:
# --- CODE CELL ---
def best_first(grid, start, end):
    """
    Best-First Search (Greedy Best-First).
    Priority = heuristic only. No g-score tracking.
    """
    # open_set entries: (h_score, node)
    open_set = [(heuristic(start, end), start)]
    seen = set()
    parent = {start: None}
    visited = []

    while open_set:
        open_set.sort(key=lambda x: x[0])  # Sort by heuristic only
        _, curr = open_set.pop(0)

        if curr in seen:
            continue

        seen.add(curr)
        visited.append(curr)

        if curr == end:
            return visited, reconstruct_path(parent, end)

        for nb in neighbors(grid, curr):
            if nb not in seen:
                parent[nb] = curr
                open_set.append((heuristic(nb, end), nb))

    return visited, []


g = make_grid()
v, p = best_first(g, (0,0), (10,20))
print(f"Best-First visited {len(v)} cells, path = {len(p)} steps")

Best-First visited 31 cells, path = 31 steps


# 11. Algorithm 6 - MiniMax
### How MiniMax works
MiniMax is an **adversarial search algorithm** from game theory. 
Here we adapt it for pathfinding:
- **Maximizing player** : tries to move away from the goal (adversary/obstacles)
- **Minimizing player** : tries to move toward the goal (our pathfinder)
The algorithm alternates between these two "players" at each depth level.

Minimax uses recursion instead of an explicit stack/queue.

In [12]:
def minimax(grid, start, end, depth=5):
    """
    MiniMax Search adapted for pathfinding.
    Alternates between maximizing (adversarial) and minimizing (goal-seeking) moves.

    depth: how many levels deep to search
    is_max: True = maximizer's turn (tries to increase h — move away from goal)
            False = minimizer's turn (tries to decrease h — move toward goal)
    """
    seen = set()
    parent = {start: None}
    visited = []

    def mm(node, d, is_max):
        """Recursive inner function."""
        # Base cases: visited before, depth exhausted, or stuck
        if node in seen or d == 0:
            return heuristic(node, end)  # Return h-score as evaluation

        seen.add(node)
        visited.append(node)

        if node == end:
            return 0  # Score of 0 = goal reached (best for minimizer)

        nbs = [n for n in neighbors(grid, node) if n not in seen]
        if not nbs:
            return heuristic(node, end)  # Dead end

        if is_max:
            # Maximizer: pick move that gives HIGHEST score (worst for us)
            best_val = -float("inf")
            for nb in nbs:
                val = mm(nb, d - 1, False)  # Opponent's turn next
                if val > best_val:
                    best_val = val
                    parent[nb] = node
            return best_val
        else:
            # Minimizer: pick move that gives LOWEST score (toward goal)
            best_val = float("inf")
            for nb in nbs:
                val = mm(nb, d - 1, True)   # Adversary's turn next
                if val < best_val:
                    best_val = val
                    parent[nb] = node
            return best_val

    mm(start, depth, True)  # Start with maximizer (adversarial exploration)
    return visited, reconstruct_path(parent, end)


g = make_grid()
v, p = minimax(g, (0,0), (5,5))
print(f"MiniMax visited {len(v)} cells, path = {len(p)} steps")

MiniMax visited 15 cells, path = 0 steps


# 11. Algorithm Dispatcher
### Two Helper functions:
- **run_algorithm** : maps the selected algorithm name to its function
- **apply_case_mode** : modifies the grid based on Best/Average/Worst case:
   - Best Case: Removes all walls (open grid, algorithm performs best) 
   - Average Case: Uses the grid a is (whatever the user drew)
   - Worst Case: randomaly adds ~35% extra walls (harder for algorithm)

In [13]:
import random

def run_algorithm(name, grid, start, end):
    """
    Dispatcher: maps algorithm name → function.
    Returns (visited_list, path_list) from the chosen algorithm.
    """
    if name == "BFS":           return bfs(grid, start, end)
    if name == "DFS":           return dfs(grid, start, end)
    if name == "A*":            return astar(grid, start, end)
    if name == "Hill Climbing": return hill_climbing(grid, start, end)
    if name == "Best First":    return best_first(grid, start, end)
    if name == "MiniMax":       return minimax(grid, start, end)
    return [], []


def apply_case_mode(base_grid, mode, start, end):
    """
    Returns a COPY of base_grid modified for the given case mode.
    We copy first with [row[:] for row in base_grid] so the original is untouched.
    """
    grid = [row[:] for row in base_grid]  # Deep copy of 2D list

    if mode == "Best Case":
        # Remove all walls — maximum open space
        for r in range(ROWS):
            for c in range(COLS):
                if grid[r][c] == "wall":
                    grid[r][c] = "empty"

    if mode == "Worst Case":
        # Randomly add more walls (35% chance per empty cell)
        for r in range(ROWS):
            for c in range(COLS):
                if (r, c) == start or (r, c) == end:
                    continue         # Never wall over start/end
                if random.random() < 0.35:
                    grid[r][c] = "wall"

    # Average Case: return grid unchanged
    return grid


# Test:
g = make_grid()
g[0][1] = g[0][2] = "wall"  # add some walls
best  = apply_case_mode(g, "Best Case",  (0,0), (19,29))
worst = apply_case_mode(g, "Worst Case", (0,0), (19,29))

wall_count_base  = sum(1 for r in g     for c in r if c == "wall")
wall_count_best  = sum(1 for r in best  for c in r if c == "wall")
wall_count_worst = sum(1 for r in worst for c in r if c == "wall")

print(f"Base walls: {wall_count_base}, Best-case walls: {wall_count_best}, Worst-case walls: {wall_count_worst}")

Base walls: 2, Best-case walls: 0, Worst-case walls: 220


# 12. Visualizer Class 


In [14]:
import pygame

class Visualizer:
    def __init__(self):
        # Pygame initialization
        pygame.init()  # Must be called before any other pygame function
        pygame.display.set_caption("AI Path Finding Visualizer - Python")

        # Create the window surface (what we draw on)
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))

        # Clock manages frame rate (we target 60 FPS)
        self.clock = pygame.time.Clock()

        # Fonts — pygame.font.SysFont("font_name", size, bold=False)
        # Different sizes for different UI elements
        self.title = pygame.font.SysFont("Avenir Next", 22, bold=True)
        self.font  = pygame.font.SysFont("Avenir Next", 18, bold=True)
        self.small = pygame.font.SysFont("Segoe UI",    14)
        self.tiny  = pygame.font.SysFont("Consolas",    13)   # monospace for heuristic values
        self.badge = pygame.font.SysFont("Segoe UI",    12, bold=True)

        # Application state 
        self.grid       = make_grid()          # 2D list of cell states
        self.start      = (5, 3)               # Starting position (row, col)
        self.end        = (14, 26)             # Ending position
        self.tool       = TOOL_WALL            # Active drawing tool
        self.algorithm  = "A*"                 # Currently selected algorithm
        self.case_mode  = "Average Case"       # Best/Average/Worst
        self.speed      = 18                   # Animation speed (1-20)
        self.show_values = True                # Show heuristic values on cells

        # Input state
        self.mouse_down = False                # Is left mouse button held?

        # Animation state 
        self.running       = False             # Is animation playing?
        self.visited_order = []                # Cells explored, in order
        self.path          = []                # Final path cells
        self.visited_index = 0                 # Next visited cell to reveal
        self.path_index    = 0                 # Next path cell to reveal
        self.path_started  = False             # Has path animation begun?
        self.last_tick     = 0                 # Time of last animation frame
        self.path_start_at = 0                 # When to start path animation
        self.stats         = None              # Dict of results metrics

        # Rendering state 
        # cell_states: maps (r,c) → color key for animated cells
        self.cell_states = {}

        # ui_buttons: list of {rect, action, value} — rebuilt each frame
        self.ui_buttons = []

print("Visualizer.__init__ defined — explains setup logic")

pygame 2.6.1 (SDL 2.28.4, Python 3.13.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
Visualizer.__init__ defined — explains setup logic


# 13. Visualizer Class - Event Handling
### Pygame event system
Every user action generates an event in Pygame's event queue.We drain this queue each frame in handle_events().

In [15]:
# (Explanatory snippet — part of the Visualizer class)

def clear_visualization(self):
    """Reset animation state but keep the grid (walls/start/end intact)."""
    self.running       = False
    self.visited_order = []
    self.path          = []
    self.visited_index = 0
    self.path_index    = 0
    self.path_started  = False
    self.last_tick     = 0
    self.path_start_at = 0
    self.stats         = None
    self.cell_states   = {}  # Clear colored cells


def add_button(self, rect, action, value=None):
    """Register a clickable button. Called during drawing phase."""
    self.ui_buttons.append({"rect": rect, "action": action, "value": value})


def handle_ui_click(self, pos):
    """
    Check if mouse click lands on any registered button.
    We iterate REVERSED so the topmost (last drawn) button wins on overlap.
    Returns True if a button was clicked.
    """
    for btn in reversed(self.ui_buttons):
        if btn["rect"].collidepoint(pos):  # .collidepoint = "is point inside rect?"
            action, value = btn["action"], btn["value"]

            if action == "algo":         self.algorithm  = value
            elif action == "mode":       self.case_mode  = value
            elif action == "tool":       self.tool       = value
            elif action == "visualize":
                if not self.running:     self.visualize()
            elif action == "clear":      self.clear_visualization()
            elif action == "reset":      self.reset_grid()
            elif action == "speed_inc":  self.speed = min(20, self.speed + 1)
            elif action == "speed_dec":  self.speed = max(1,  self.speed - 1)
            elif action == "toggle_values": self.show_values = not self.show_values

            return True
    return False


def handle_events(self):
    """
    Main event loop — drains pygame's event queue each frame.
    Handles: quit, keyboard shortcuts, mouse clicks, mouse drag.
    """
    for event in pygame.event.get():

        # ── QUIT ───────────────────────────────────────────────────
        if event.type == pygame.QUIT:
            pygame.quit()
            import sys; sys.exit(0)

        # ── KEYBOARD ───────────────────────────────────────────────
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE: pygame.quit(); import sys; sys.exit(0)
            if event.key == pygame.K_SPACE and not self.running: self.visualize()
            if event.key == pygame.K_c:  self.clear_visualization()
            if event.key == pygame.K_r:  self.reset_grid()
            if event.key == pygame.K_v:  self.show_values = not self.show_values
            if event.key == pygame.K_s:  self.tool = TOOL_START
            if event.key == pygame.K_e:  self.tool = TOOL_END
            if event.key == pygame.K_d:  self.tool = TOOL_WALL
            if event.key == pygame.K_b:  self.case_mode = "Best Case"
            if event.key == pygame.K_a:  self.case_mode = "Average Case"
            if event.key == pygame.K_w:  self.case_mode = "Worst Case"
            if event.key == pygame.K_UP:   self.speed = min(20, self.speed + 1)
            if event.key == pygame.K_DOWN: self.speed = max(1,  self.speed - 1)

            # Keys 1-6 → select algorithm 1-6
            if pygame.K_1 <= event.key <= pygame.K_6:
                self.algorithm = ALGORITHMS[event.key - pygame.K_1]

        # ── MOUSE BUTTON DOWN ──────────────────────────────────────
        if event.type == pygame.MOUSEBUTTONDOWN:
            if event.button == 1 and self.handle_ui_click(event.pos):
                continue  # UI click handled; skip grid interaction

            rc = self.cell_at_mouse(event.pos)  # Which grid cell?
            if rc is None: continue

            self.mouse_down = True
            r, c = rc
            if event.button == 1: self.toggle_cell(r, c)  # Left click
            if event.button == 3: self.erase_cell(r, c)   # Right click = erase

        if event.type == pygame.MOUSEBUTTONUP:
            self.mouse_down = False

        # ── MOUSE DRAG ─────────────────────────────────────────────
        if event.type == pygame.MOUSEMOTION and self.mouse_down:
            rc = self.cell_at_mouse(event.pos)
            if rc is None: continue
            r, c = rc
            # Drag with left button + wall tool = paint walls
            if pygame.mouse.get_pressed()[0] and self.tool == TOOL_WALL:
                if (r, c) != self.start and (r, c) != self.end:
                    self.grid[r][c] = "wall"
                    self.clear_visualization()
            # Drag with right button = erase
            if pygame.mouse.get_pressed()[2]:
                self.erase_cell(r, c)


print("Event handling functions defined")

Event handling functions defined


# 14. Visualizer Class - Drawing

In [16]:
# Drawing helper methods (part of Visualizer class)

def draw_chip(self, x, y, w, h, text, active=False, accent=(43, 52, 64)):
    """
    Draw a pill-shaped chip/button.
    active=True → filled with accent color (selected state)
    active=False → white background (unselected state)
    border_radius=999 makes a rectangle with fully rounded ends (pill shape).
    """
    bg     = COLORS["card"] if not active else accent
    fg     = COLORS["chip_text"] if not active else (255, 255, 255)
    border = COLORS["border_soft"] if not active else accent
    rect   = pygame.Rect(x, y, w, h)
    pygame.draw.rect(self.screen, bg,     rect, border_radius=999)
    pygame.draw.rect(self.screen, border, rect, width=1, border_radius=999)
    label  = self.badge.render(text, True, fg)
    # Center the text inside the chip:
    self.screen.blit(label, (x + (w - label.get_width()) // 2,
                             y + (h - label.get_height()) // 2))


def draw_click_button(self, x, y, w, h, text, action, value=None, active=False, accent=(43, 52, 64)):
    """
    Draw a chip AND register it as a clickable button.
    This is the main button-drawing function used everywhere.
    """
    rect = pygame.Rect(x, y, w, h)
    self.draw_chip(x, y, w, h, text, active=active, accent=accent)
    self.add_button(rect, action, value)  # Register for click detection


def draw_stat_card(self, x, y, w, h, label, value, value_color=None):
    """
    Draw a metric card with a small label on top and a value below.
    Used in the side panel for: Status, Nodes, Path length, Time.
    """
    rect = pygame.Rect(x, y, w, h)
    pygame.draw.rect(self.screen, COLORS["card"],        rect, border_radius=12)
    pygame.draw.rect(self.screen, COLORS["border_soft"], rect, width=1, border_radius=12)
    self.screen.blit(self.badge.render(label, True, COLORS["muted"]),
                     (x + 10, y + 7))
    val_surface = self.small.render(value, True, value_color or COLORS["text"])
    self.screen.blit(val_surface, (x + 10, y + 26))


def cell_color(self, r, c):
    """
    Determines the color to draw for cell (r, c).
    Priority order:
    1. Start/End → always green/orange
    2. Animated cell (visited or path) → from cell_states dict
    3. Wall → dark
    4. Empty → light cream
    """
    if (r, c) == self.start:   return COLORS["start"]
    if (r, c) == self.end:     return COLORS["end"]
    if (r, c) in self.cell_states: return COLORS[self.cell_states[(r, c)]]
    if self.grid[r][c] == "wall":  return COLORS["wall"]
    return COLORS["empty"]


print("Drawing helper methods defined")

Drawing helper methods defined


In [17]:
# draw_header, draw_grid, draw_side_panel, draw_footer (part of Visualizer)

def draw_header(self):
    """
    Draws the top toolbar:
    - App title (left)
    - Row 1: Algorithm selector chips (BFS, DFS, A*, ...)
    - Row 2: Case mode chips + Tool chips + Action buttons

    IMPORTANT: self.ui_buttons is RESET here at the start of each draw cycle.
    Button positions depend on text width, so we compute dynamically.
    """
    self.ui_buttons = []  # ← Clear button list each frame (rebuilt below)

    pygame.draw.rect(self.screen, COLORS["panel"],
                     (0, 0, WINDOW_WIDTH, HEADER_HEIGHT))
    pygame.draw.line(self.screen, COLORS["border"],
                     (0, HEADER_HEIGHT - 1), (WINDOW_WIDTH, HEADER_HEIGHT - 1), 1)

    # Title text
    self.screen.blit(self.title.render("AI Pathfinding Visualizer", True, COLORS["text"]),
                     (14, 10))
    self.screen.blit(self.badge.render("PYTHON EDITION • 6 ALGORITHMS", True, COLORS["muted"]),
                     (16, 40))

    # Row 1: Algorithm buttons
    row1_y, x = 10, 350
    for algo in ALGORITHMS:
        w = max(66, 14 + self.badge.size(algo)[0] + 14)  # Dynamic width
        self.draw_click_button(x, row1_y, w, 24, algo, "algo", value=algo,
                               active=(self.algorithm == algo), accent=(43, 52, 64))
        x += w + 6

    # Row 2: Mode buttons + Tool buttons + Action buttons
    row2_y, x = 40, 350
    for mode in MODES:
        w = max(94, 14 + self.badge.size(mode)[0] + 14)
        self.draw_click_button(x, row2_y, w, 24, mode, "mode", value=mode,
                               active=(self.case_mode == mode), accent=CASE_COLORS[mode])
        x += w + 6

    for tool_id, label in ((TOOL_WALL, "Wall"), (TOOL_START, "Start"), (TOOL_END, "End")):
        self.draw_click_button(x, row2_y, 68, 24, label, "tool", value=tool_id,
                               active=(self.tool == tool_id), accent=(64, 72, 84))
        x += 74

    self.draw_click_button(x, row2_y, 86, 24, "Visualize", "visualize",
                           active=not self.running, accent=(29, 158, 117))
    x += 92
    self.draw_click_button(x, row2_y, 60, 24, "Clear", "clear")
    x += 66
    self.draw_click_button(x, row2_y, 60, 24, "Reset", "reset")


def draw_grid(self):
    """
    Draws the main 30×20 cell grid.
    Each cell is a colored rectangle + border.
    Start/End cells show 'S' / 'E' labels.
    If show_values=True, displays the Manhattan distance to End in each cell.
    """
    for r in range(ROWS):
        for c in range(COLS):
            x = c * CELL
            y = HEADER_HEIGHT + r * CELL
            rect = pygame.Rect(x, y, CELL, CELL)

            pygame.draw.rect(self.screen, self.cell_color(r, c), rect)
            pygame.draw.rect(self.screen, COLORS["border_soft"], rect, 1)

            if (r, c) == self.start:
                txt = self.small.render("S", True, (255, 255, 255))
                self.screen.blit(txt, (x + 10, y + 6))
            elif (r, c) == self.end:
                txt = self.small.render("E", True, (255, 255, 255))
                self.screen.blit(txt, (x + 10, y + 6))
            elif self.show_values and self.grid[r][c] != "wall":
                # Show heuristic value (Manhattan distance to end)
                v = str(heuristic((r, c), self.end))
                txt = self.tiny.render(v, True, (94, 104, 116))
                txt_rect = txt.get_rect(center=(x + CELL // 2, y + CELL // 2))
                self.screen.blit(txt, txt_rect)


print("draw_header and draw_grid defined")

draw_header and draw_grid defined


# 15. Visualizer Class - Animation Loop
The animation reveals cells one-at-a-time using time-based ticks rather than frame-based:
every n milliseconds- reveals one more visited cell 
After all visited cells shown - wait a moment - reveal path cells
**visualize()** - runs algorithm once 
This function:
   1. Applies the case mode (modifying grid)
   2. Runs the chosen algorithm and records timing
   3. Set animation state to start revealing cells.
   

In [18]:
import time

def speed_delay(self):
    """
    Converts speed (1-20) to milliseconds per cell reveal.
    speed=20 → delay=1ms (fastest)
    speed=1  → delay=20ms (slowest)
    """
    return max(1, 21 - self.speed)


def visualize(self):
    """
    Run the selected algorithm and set up animation state.
    Called when user presses SPACE or clicks 'Visualize'.
    """
    self.clear_visualization()

    # Apply case mode (may modify grid)
    case_grid = apply_case_mode(self.grid, self.case_mode, self.start, self.end)
    self.grid = case_grid  # Replace grid (Worst Case adds walls)

    # Run algorithm and measure time
    t0 = time.perf_counter()
    self.visited_order, self.path = run_algorithm(
        self.algorithm, case_grid, self.start, self.end)
    t1 = time.perf_counter()

    # Start animation
    self.running   = True
    self.last_tick = pygame.time.get_ticks()

    # Path animation starts AFTER all visited cells are shown + 50ms gap
    self.path_start_at = (self.last_tick +
                          len(self.visited_order) * self.speed_delay() + 50)

    # Determine path color based on case mode
    path_color = {"Best Case": "best_path",
                  "Worst Case": "worst_path"}.get(self.case_mode, "avg_path")

    self.stats = {
        "algo":       self.algorithm,
        "visited":    len(self.visited_order),
        "path_len":   len(self.path),
        "found":      len(self.path) > 0,
        "time_ms":    f"{(t1 - t0) * 1000:.2f}",
        "case_mode":  self.case_mode,
        "path_color": path_color,
    }


def animate(self):
    """
    Called every frame. Reveals one cell at a time based on elapsed time.

    Phase 1: Reveal visited cells one-by-one
    Phase 2: Reveal path cells one-by-one (at 2× slower speed)
    Phase 3: Animation complete → self.running = False
    """
    if not self.running:
        return

    now   = pygame.time.get_ticks()  # Current time in ms
    delay = self.speed_delay()

    # ── Phase 1: Reveal visited cells ──────────────────────────────
    if self.visited_index < len(self.visited_order):
        if now - self.last_tick >= delay:
            node = self.visited_order[self.visited_index]
            self.cell_states[node] = "visited"   # Mark cell as visited
            self.visited_index += 1
            self.last_tick = now
        return  # Don't proceed to path phase yet

    # ── Phase 2: Wait, then reveal path cells ──────────────────────
    if not self.path_started and now >= self.path_start_at:
        self.path_started = True
        self.last_tick    = now

    if self.path_started and self.path_index < len(self.path):
        if now - self.last_tick >= delay * 2:  # Path reveals 2× slower
            node = self.path[self.path_index]
            self.cell_states[node] = self.stats["path_color"]
            self.path_index += 1
            self.last_tick = now
        return

    # ── Phase 3: Done ──────────────────────────────────────────────
    self.running = False
print("Animation methods defined")

Animation methods defined


# 16. Save and run the complete file
The cell below writes every section above into a single pathfinding_visualizer.py.

In [21]:
#  Save the complete project to pathfinding_visualizer.py

code = '''
import random
import sys
import time
from collections import deque

import pygame

#  CONSTANTS
ROWS = 20
COLS = 30
CELL = 28
PANEL_WIDTH   = 330
HEADER_HEIGHT = 72
FOOTER_HEIGHT = 86
GRID_WIDTH    = COLS * CELL
GRID_HEIGHT   = ROWS * CELL
WINDOW_WIDTH  = GRID_WIDTH + PANEL_WIDTH
WINDOW_HEIGHT = HEADER_HEIGHT + GRID_HEIGHT + FOOTER_HEIGHT

#  COLORS & METADATA
    "bg": (245, 241, 230),
    "bg_alt": (233, 242, 239),
    "panel": (250, 247, 240),
    "panel_soft": (240, 235, 225),
    "card": (255, 255, 252),
    "border": (182, 172, 155),
    "border_soft": (210, 201, 187),
    "text": (28,  35,  44 ),
    "muted": (96,  104, 112),
    "chip_text": (67,  75,  83 ),
    "empty": (247, 245, 238),
    "wall": (44,  44,  42 ),
    "start": (29,  158, 117),
    "end": (216, 90,  48 ),
    "visited": (181, 212, 244),
    "best_path": (239, 159, 39 ),
    "avg_path": (127, 119, 221),
    "worst_path": (226, 75,  74 ),
}

CASE_COLORS = {
    "Best Case":    COLORS["start"],
    "Average Case": COLORS["avg_path"],
    "Worst Case":   COLORS["worst_path"],
}

ALGO_BLURB = {
    "BFS":           "Explores level-by-level and guarantees shortest path on unweighted grids.",
    "DFS":           "Dives deep into one branch first and backtracks when blocked.",
    "A*":            "Balances known cost and estimated distance for efficient optimal paths.",
    "Hill Climbing": "Greedy local search moving to better immediate neighbors.",
    "Best First":    "Expands the most promising node according to heuristic only.",
    "MiniMax":       "Adversarial-style exploration with alternating maximize/minimize choices.",
}

TIME_COMPLEXITY = {
    "BFS":           ("O(1)",   "O(V+E)",   "O(V+E)"),
    "DFS":           ("O(1)",   "O(V+E)",   "O(V+E)"),
    "A*":            ("O(b^d)", "O(b^d)",   "O(b^d)"),
    "Hill Climbing": ("O(b*m)", "O(b*m)",   "O(b*m)"),
    "Best First":    ("O(b^m)", "O(b^m)",   "O(b^m)"),
    "MiniMax":       ("O(b^m)", "O(b^m)",   "O(b^m)"),
}

ALGORITHMS = ["BFS", "DFS", "A*", "Hill Climbing", "Best First", "MiniMax"]
MODES      = ["Best Case", "Average Case", "Worst Case"]
TOOL_WALL  = "wall"
TOOL_START = "start"
TOOL_END   = "end"



#  GRID UTILITIES
def make_grid():
    return [["empty" for _ in range(COLS)] for _ in range(ROWS)]


def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def neighbors(grid, node):
    r, c = node
    out = []
    for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != "wall":
            out.append((nr, nc))
    return out


def reconstruct_path(parent, end):
    path = []
    curr = end
    while curr is not None:
        path.append(curr)
        curr = parent.get(curr)
    path.reverse()
    return path if len(path) > 1 else []

#  ALGORITHMS
def bfs(grid, start, end):
    q = deque([start])
    seen = {start}
    parent = {start: None}
    visited = []
    while q:
        curr = q.popleft()
        visited.append(curr)
        if curr == end:
            return visited, reconstruct_path(parent, end)
        for nb in neighbors(grid, curr):
            if nb not in seen:
                seen.add(nb)
                parent[nb] = curr
                q.append(nb)
    return visited, []


def dfs(grid, start, end):
    stack = [start]
    seen = set()
    parent = {start: None}
    visited = []
    while stack:
        curr = stack.pop()
        if curr in seen:
            continue
        seen.add(curr)
        visited.append(curr)
        if curr == end:
            return visited, reconstruct_path(parent, end)
        for nb in neighbors(grid, curr):
            if nb not in seen:
                parent[nb] = curr
                stack.append(nb)
    return visited, []


def astar(grid, start, end):
    open_set = [(heuristic(start, end), 0, start)]
    seen = set()
    parent = {start: None}
    g_score = {start: 0}
    visited = []
    while open_set:
        open_set.sort(key=lambda x: x[0])
        _, g_curr, curr = open_set.pop(0)
        if curr in seen:
            continue
        seen.add(curr)
        visited.append(curr)
        if curr == end:
            return visited, reconstruct_path(parent, end)
        for nb in neighbors(grid, curr):
            tentative = g_curr + 1
            if tentative < g_score.get(nb, float("inf")):
                g_score[nb] = tentative
                parent[nb] = curr
                h = heuristic(nb, end)
                open_set.append((tentative + h, tentative, nb))
    return visited, []


def hill_climbing(grid, start, end):
    curr = start
    seen = set()
    parent = {start: None}
    visited = []
    while curr is not None:
        if curr in seen:
            break
        seen.add(curr)
        visited.append(curr)
        if curr == end:
            return visited, reconstruct_path(parent, end)
        nbs = [n for n in neighbors(grid, curr) if n not in seen]
        nbs.sort(key=lambda n: heuristic(n, end))
        if nbs and heuristic(nbs[0], end) < heuristic(curr, end):
            nxt = nbs[0]
            parent[nxt] = curr
            curr = nxt
        else:
            break
    return visited, []


def best_first(grid, start, end):
    open_set = [(heuristic(start, end), start)]
    seen = set()
    parent = {start: None}
    visited = []
    while open_set:
        open_set.sort(key=lambda x: x[0])
        _, curr = open_set.pop(0)
        if curr in seen:
            continue
        seen.add(curr)
        visited.append(curr)
        if curr == end:
            return visited, reconstruct_path(parent, end)
        for nb in neighbors(grid, curr):
            if nb not in seen:
                parent[nb] = curr
                open_set.append((heuristic(nb, end), nb))
    return visited, []


def minimax(grid, start, end, depth=5):
    seen = set()
    parent = {start: None}
    visited = []
    def mm(node, d, is_max):
        if node in seen or d == 0:
            return heuristic(node, end)
        seen.add(node)
        visited.append(node)
        if node == end:
            return 0
        nbs = [n for n in neighbors(grid, node) if n not in seen]
        if not nbs:
            return heuristic(node, end)
        if is_max:
            best_val = -float("inf")
            for nb in nbs:
                val = mm(nb, d - 1, False)
                if val > best_val:
                    best_val = val
                    parent[nb] = node
            return best_val
        best_val = float("inf")
        for nb in nbs:
            val = mm(nb, d - 1, True)
            if val < best_val:
                best_val = val
                parent[nb] = node
        return best_val
    mm(start, depth, True)
    return visited, reconstruct_path(parent, end)


def run_algorithm(name, grid, start, end):
    if name == "BFS":           return bfs(grid, start, end)
    if name == "DFS":           return dfs(grid, start, end)
    if name == "A*":            return astar(grid, start, end)
    if name == "Hill Climbing": return hill_climbing(grid, start, end)
    if name == "Best First":    return best_first(grid, start, end)
    if name == "MiniMax":       return minimax(grid, start, end)
    return [], []


def apply_case_mode(base_grid, mode, start, end):
    grid = [row[:] for row in base_grid]
    if mode == "Best Case":
        for r in range(ROWS):
            for c in range(COLS):
                if grid[r][c] == "wall":
                    grid[r][c] = "empty"
    if mode == "Worst Case":
        for r in range(ROWS):
            for c in range(COLS):
                if (r, c) == start or (r, c) == end:
                    continue
                if random.random() < 0.35:
                    grid[r][c] = "wall"
    return grid


#  VISUALIZER CLASS
class Visualizer:
    def __init__(self):
        pygame.init()
        pygame.display.set_caption("AI Path Finding Visualizer - Python")
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        self.clock  = pygame.time.Clock()
        self.title  = pygame.font.SysFont("Avenir Next", 22, bold=True)
        self.font   = pygame.font.SysFont("Avenir Next", 18, bold=True)
        self.small  = pygame.font.SysFont("Segoe UI",    14)
        self.tiny   = pygame.font.SysFont("Consolas",    13)
        self.badge  = pygame.font.SysFont("Segoe UI",    12, bold=True)

        self.grid        = make_grid()
        self.start       = (5, 3)
        self.end         = (14, 26)
        self.tool        = TOOL_WALL
        self.algorithm   = "A*"
        self.case_mode   = "Average Case"
        self.speed       = 18
        self.show_values = True
        self.mouse_down  = False
        self.running     = False
        self.visited_order = []
        self.path          = []
        self.visited_index = 0
        self.path_index    = 0
        self.path_started  = False
        self.last_tick     = 0
        self.path_start_at = 0
        self.stats         = None
        self.cell_states   = {}
        self.ui_buttons    = []

    # State management
    def clear_visualization(self):
        self.running = False
        self.visited_order = []
        self.path = []
        self.visited_index = self.path_index = 0
        self.path_started = False
        self.last_tick = self.path_start_at = 0
        self.stats = None
        self.cell_states = {}

    def reset_grid(self):
        self.clear_visualization()
        self.grid = make_grid()

    def speed_delay(self):
        return max(1, 21 - self.speed)

    def toggle_cell(self, r, c):
        if self.tool == TOOL_START:
            self.start = (r, c); self.clear_visualization(); return
        if self.tool == TOOL_END:
            self.end = (r, c); self.clear_visualization(); return
        if (r, c) == self.start or (r, c) == self.end:
            return
        self.grid[r][c] = "wall" if self.grid[r][c] != "wall" else "empty"
        self.clear_visualization()

    def erase_cell(self, r, c):
        if (r, c) in (self.start, self.end): return
        self.grid[r][c] = "empty"
        self.clear_visualization()

    def cell_at_mouse(self, pos):
        x, y = pos
        y -= HEADER_HEIGHT
        if x < 0 or x >= GRID_WIDTH or y < 0 or y >= GRID_HEIGHT: return None
        return (y // CELL, x // CELL)

    # Core run
    def visualize(self):
        self.clear_visualization()
        case_grid = apply_case_mode(self.grid, self.case_mode, self.start, self.end)
        self.grid = case_grid
        t0 = time.perf_counter()
        self.visited_order, self.path = run_algorithm(self.algorithm, case_grid, self.start, self.end)
        t1 = time.perf_counter()
        self.running = True
        self.last_tick = pygame.time.get_ticks()
        self.path_start_at = self.last_tick + len(self.visited_order) * self.speed_delay() + 50
        path_color = {"Best Case": "best_path", "Worst Case": "worst_path"}.get(self.case_mode, "avg_path")
        self.stats = {
            "algo": self.algorithm, "visited": len(self.visited_order),
            "path_len": len(self.path), "found": len(self.path) > 0,
            "time_ms": f"{(t1-t0)*1000:.2f}", "case_mode": self.case_mode,
            "path_color": path_color,
        }

    def animate(self):
        if not self.running: return
        now, delay = pygame.time.get_ticks(), self.speed_delay()
        if self.visited_index < len(self.visited_order):
            if now - self.last_tick >= delay:
                self.cell_states[self.visited_order[self.visited_index]] = "visited"
                self.visited_index += 1
                self.last_tick = now
            return
        if not self.path_started and now >= self.path_start_at:
            self.path_started = True
            self.last_tick = now
        if self.path_started and self.path_index < len(self.path):
            if now - self.last_tick >= delay * 2:
                self.cell_states[self.path[self.path_index]] = self.stats["path_color"]
                self.path_index += 1
                self.last_tick = now
            return
        self.running = False

    # UI button system
    def add_button(self, rect, action, value=None):
        self.ui_buttons.append({"rect": rect, "action": action, "value": value})

    def handle_ui_click(self, pos):
        for btn in reversed(self.ui_buttons):
            if btn["rect"].collidepoint(pos):
                a, v = btn["action"], btn["value"]
                if a == "algo":          self.algorithm = v
                elif a == "mode":        self.case_mode = v
                elif a == "tool":        self.tool = v
                elif a == "visualize" and not self.running: self.visualize()
                elif a == "clear":       self.clear_visualization()
                elif a == "reset":       self.reset_grid()
                elif a == "speed_inc":   self.speed = min(20, self.speed + 1)
                elif a == "speed_dec":   self.speed = max(1,  self.speed - 1)
                elif a == "toggle_values": self.show_values = not self.show_values
                return True
        return False

    # Events
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit(); sys.exit(0)
            if event.type == pygame.KEYDOWN:
                k = event.key
                if k == pygame.K_ESCAPE: pygame.quit(); sys.exit(0)
                if k == pygame.K_SPACE and not self.running: self.visualize()
                if k == pygame.K_c: self.clear_visualization()
                if k == pygame.K_r: self.reset_grid()
                if k == pygame.K_v: self.show_values = not self.show_values
                if k == pygame.K_s: self.tool = TOOL_START
                if k == pygame.K_e: self.tool = TOOL_END
                if k == pygame.K_d: self.tool = TOOL_WALL
                if k == pygame.K_b: self.case_mode = "Best Case"
                if k == pygame.K_a: self.case_mode = "Average Case"
                if k == pygame.K_w: self.case_mode = "Worst Case"
                if k == pygame.K_UP:   self.speed = min(20, self.speed + 1)
                if k == pygame.K_DOWN: self.speed = max(1,  self.speed - 1)
                if pygame.K_1 <= k <= pygame.K_6:
                    self.algorithm = ALGORITHMS[k - pygame.K_1]
            if event.type == pygame.MOUSEBUTTONDOWN:
                if event.button == 1 and self.handle_ui_click(event.pos): continue
                rc = self.cell_at_mouse(event.pos)
                if rc is None: continue
                self.mouse_down = True
                r, c = rc
                if event.button == 1: self.toggle_cell(r, c)
                if event.button == 3: self.erase_cell(r, c)
            if event.type == pygame.MOUSEBUTTONUP:
                self.mouse_down = False
            if event.type == pygame.MOUSEMOTION and self.mouse_down:
                rc = self.cell_at_mouse(event.pos)
                if rc is None: continue
                r, c = rc
                if pygame.mouse.get_pressed()[0] and self.tool == TOOL_WALL:
                    if (r, c) not in (self.start, self.end):
                        self.grid[r][c] = "wall"; self.clear_visualization()
                if pygame.mouse.get_pressed()[2]:
                    self.erase_cell(r, c)

    # Drawing helpers
    def cell_color(self, r, c):
        if (r, c) == self.start: return COLORS["start"]
        if (r, c) == self.end:   return COLORS["end"]
        if (r, c) in self.cell_states: return COLORS[self.cell_states[(r, c)]]
        if self.grid[r][c] == "wall": return COLORS["wall"]
        return COLORS["empty"]

    def draw_chip(self, x, y, w, h, text, active=False, accent=(43, 52, 64)):
        bg     = COLORS["card"] if not active else accent
        fg     = COLORS["chip_text"] if not active else (255, 255, 255)
        border = COLORS["border_soft"] if not active else accent
        rect   = pygame.Rect(x, y, w, h)
        pygame.draw.rect(self.screen, bg,     rect, border_radius=999)
        pygame.draw.rect(self.screen, border, rect, width=1, border_radius=999)
        label  = self.badge.render(text, True, fg)
        self.screen.blit(label, (x + (w - label.get_width()) // 2,
                                 y + (h - label.get_height()) // 2))

    def draw_click_button(self, x, y, w, h, text, action, value=None, active=False, accent=(43, 52, 64)):
        self.draw_chip(x, y, w, h, text, active=active, accent=accent)
        self.add_button(pygame.Rect(x, y, w, h), action, value)

    def draw_stat_card(self, x, y, w, h, label, value, value_color=None):
        rect = pygame.Rect(x, y, w, h)
        pygame.draw.rect(self.screen, COLORS["card"],        rect, border_radius=12)
        pygame.draw.rect(self.screen, COLORS["border_soft"], rect, width=1, border_radius=12)
        self.screen.blit(self.badge.render(label, True, COLORS["muted"]), (x + 10, y + 7))
        self.screen.blit(self.small.render(value, True, value_color or COLORS["text"]), (x + 10, y + 26))

    def draw_wrapped_text(self, text, x, y, max_width, color):
        words, line, line_h = text.split(" "), "", self.small.get_linesize()
        for word in words:
            candidate = (line + " " + word).strip()
            if self.small.size(candidate)[0] <= max_width:
                line = candidate; continue
            self.screen.blit(self.small.render(line, True, color), (x, y))
            y += line_h + 2; line = word
        if line: self.screen.blit(self.small.render(line, True, color), (x, y))

    # Draw sections
    def draw_background(self):
        self.screen.fill(COLORS["bg"])
        halo_a = pygame.Surface((420, 420), pygame.SRCALPHA)
        pygame.draw.circle(halo_a, (127, 119, 221, 28), (210, 210), 205)
        self.screen.blit(halo_a, (-100, -70))
        halo_b = pygame.Surface((500, 500), pygame.SRCALPHA)
        pygame.draw.circle(halo_b, (29, 158, 117, 22), (250, 250), 230)
        self.screen.blit(halo_b, (WINDOW_WIDTH - 430, -150))

    def draw_header(self):
        self.ui_buttons = []
        pygame.draw.rect(self.screen, COLORS["panel"], (0, 0, WINDOW_WIDTH, HEADER_HEIGHT))
        pygame.draw.line(self.screen, COLORS["border"],
                         (0, HEADER_HEIGHT - 1), (WINDOW_WIDTH, HEADER_HEIGHT - 1), 1)
        self.screen.blit(self.title.render("AI Pathfinding Visualizer", True, COLORS["text"]), (14, 10))
        self.screen.blit(self.badge.render("PYTHON EDITION • 6 ALGORITHMS", True, COLORS["muted"]), (16, 40))
        row1_y, x = 10, 350
        for algo in ALGORITHMS:
            w = max(66, 14 + self.badge.size(algo)[0] + 14)
            self.draw_click_button(x, row1_y, w, 24, algo, "algo", value=algo,
                                   active=(self.algorithm == algo), accent=(43, 52, 64))
            x += w + 6
        row2_y, x = 40, 350
        for mode in MODES:
            w = max(94, 14 + self.badge.size(mode)[0] + 14)
            self.draw_click_button(x, row2_y, w, 24, mode, "mode", value=mode,
                                   active=(self.case_mode == mode), accent=CASE_COLORS[mode])
            x += w + 6
        for tool_id, label in ((TOOL_WALL, "Wall"), (TOOL_START, "Start"), (TOOL_END, "End")):
            self.draw_click_button(x, row2_y, 68, 24, label, "tool", value=tool_id,
                                   active=(self.tool == tool_id), accent=(64, 72, 84))
            x += 74
        self.draw_click_button(x, row2_y, 86, 24, "Visualize", "visualize",
                               active=not self.running, accent=(29, 158, 117))
        x += 92
        self.draw_click_button(x, row2_y, 60, 24, "Clear", "clear")
        x += 66
        self.draw_click_button(x, row2_y, 60, 24, "Reset", "reset")

    def draw_grid(self):
        grid_bg = pygame.Rect(8, HEADER_HEIGHT + 8, GRID_WIDTH - 16, GRID_HEIGHT - 16)
        pygame.draw.rect(self.screen, COLORS["panel"],       grid_bg, border_radius=16)
        pygame.draw.rect(self.screen, COLORS["border_soft"], grid_bg, width=1, border_radius=16)
        for r in range(ROWS):
            for c in range(COLS):
                x, y = c * CELL, HEADER_HEIGHT + r * CELL
                rect  = pygame.Rect(x, y, CELL, CELL)
                pygame.draw.rect(self.screen, self.cell_color(r, c), rect)
                pygame.draw.rect(self.screen, COLORS["border_soft"], rect, 1)
                if (r, c) == self.start:
                    self.screen.blit(self.small.render("S", True, (255,255,255)), (x+10, y+6))
                elif (r, c) == self.end:
                    self.screen.blit(self.small.render("E", True, (255,255,255)), (x+10, y+6))
                elif self.show_values and self.grid[r][c] != "wall":
                    v   = str(heuristic((r, c), self.end))
                    txt = self.tiny.render(v, True, (94, 104, 116))
                    self.screen.blit(txt, txt.get_rect(center=(x + CELL//2, y + CELL//2)))

    def draw_side_panel(self):
        px = GRID_WIDTH
        pygame.draw.rect(self.screen, COLORS["panel_soft"], (px, HEADER_HEIGHT, PANEL_WIDTH, GRID_HEIGHT))
        pygame.draw.line(self.screen, COLORS["border"], (px, HEADER_HEIGHT), (px, HEADER_HEIGHT + GRID_HEIGHT), 1)
        card = pygame.Rect(px + 12, HEADER_HEIGHT + 12, PANEL_WIDTH - 24, GRID_HEIGHT - 24)
        pygame.draw.rect(self.screen, COLORS["panel"],       card, border_radius=16)
        pygame.draw.rect(self.screen, COLORS["border_soft"], card, width=1, border_radius=16)
        y = HEADER_HEIGHT + 26
        self.screen.blit(self.font.render("Algorithm Insight", True, COLORS["text"]), (px + 24, y))
        y += 30
        self.draw_wrapped_text(ALGO_BLURB[self.algorithm], px + 24, y, PANEL_WIDTH - 48, COLORS["muted"])
        y += 74
        self.screen.blit(self.font.render("Results", True, COLORS["text"]), (px + 24, y))
        y += 28
        if self.stats:
            status_value = "Path Found" if self.stats["found"] else "No Path"
            status_color = COLORS["start"] if self.stats["found"] else COLORS["worst_path"]
            self.draw_stat_card(px + 24,  y, 136, 58, "STATUS", status_value, value_color=status_color)
            self.draw_stat_card(px + 170, y, 136, 58, "NODES",  str(self.stats["visited"]))
            y += 66
            self.draw_stat_card(px + 24,  y, 136, 58, "PATH",   str(self.stats["path_len"]))
            self.draw_stat_card(px + 170, y, 136, 58, "TIME",   f"{self.stats[\'time_ms\']}ms")
            y += 76
            self.screen.blit(self.small.render("Time Complexity", True, COLORS["text"]), (px + 24, y))
            y += 22
            best, avg, worst = TIME_COMPLEXITY[self.stats["algo"]]
            self.draw_chip(px + 24, y, 282, 25, f"Best: {best}",    active=True, accent=COLORS["start"])
            y += 30
            self.draw_chip(px + 24, y, 282, 25, f"Average: {avg}",  active=True, accent=COLORS["avg_path"])
            y += 30
            self.draw_chip(px + 24, y, 282, 25, f"Worst: {worst}",  active=True, accent=COLORS["worst_path"])
        else:
            hint_box = pygame.Rect(px + 24, y, 282, 80)
            pygame.draw.rect(self.screen, COLORS["card"],        hint_box, border_radius=12)
            pygame.draw.rect(self.screen, COLORS["border_soft"], hint_box, width=1, border_radius=12)
            self.screen.blit(self.small.render("Press SPACE to run visualization", True, COLORS["muted"]), (px+34, y+22))
            self.screen.blit(self.small.render("Metrics will appear after run.",   True, COLORS["muted"]), (px+34, y+44))

    def draw_footer(self):
        y0 = HEADER_HEIGHT + GRID_HEIGHT
        pygame.draw.rect(self.screen, COLORS["panel"], (0, y0, WINDOW_WIDTH, FOOTER_HEIGHT))
        pygame.draw.line(self.screen, COLORS["border"], (0, y0), (WINDOW_WIDTH, y0), 1)
        self.screen.blit(self.tiny.render(
            "Tip: You can click all controls above. Keyboard shortcuts still work.",
            True, COLORS["muted"]), (14, y0 + 10))
        self.draw_click_button(WINDOW_WIDTH - 260, y0+8, 26, 24, "-", "speed_dec")
        self.draw_chip(        WINDOW_WIDTH - 230, y0+8, 74, 24, f"Speed {self.speed}")
        self.draw_click_button(WINDOW_WIDTH - 152, y0+8, 26, 24, "+", "speed_inc")
        self.draw_click_button(WINDOW_WIDTH - 122, y0+8, 108, 24,
            "Values: ON" if self.show_values else "Values: OFF",
            "toggle_values", active=self.show_values, accent=(127, 119, 221))
        legend = [("Start",   COLORS["start"]),     ("End",     COLORS["end"]),
                  ("Wall",    COLORS["wall"]),       ("Visited", COLORS["visited"]),
                  ("Best",    COLORS["best_path"]),  ("Average", COLORS["avg_path"]),
                  ("Worst",   COLORS["worst_path"])]
        x, y = 14, y0 + 36
        for label, color in legend:
            pill = pygame.Rect(x, y, 98, 28)
            pygame.draw.rect(self.screen, COLORS["card"],        pill, border_radius=999)
            pygame.draw.rect(self.screen, COLORS["border_soft"], pill, width=1, border_radius=999)
            pygame.draw.rect(self.screen, color, (x+8, y+8, 12, 12), border_radius=3)
            self.screen.blit(self.badge.render(label, True, COLORS["muted"]), (x+27, y+7))
            x += 103

    # Main loop
    def run(self):
        while True:
            self.handle_events()
            self.animate()
            self.draw_background()
            self.draw_header()
            self.draw_grid()
            self.draw_side_panel()
            self.draw_footer()
            pygame.display.flip()   # Show the rendered frame
            self.clock.tick(60)     # Cap at 60 FPS


if __name__ == "__main__":
    Visualizer().run()
'''

# Fix the escaped single quote in f-string (Python limitation in triple-quoted strings)
code = code.replace("self.stats[\'time_ms\']ms", "self.stats['time_ms']}ms")

with open("pathfinding_visualizer.py", "w") as f:
    f.write(code.strip())

print("pathfinding_visualizer.py saved!")
print(f"   Lines: {len(code.strip().splitlines())}")

pathfinding_visualizer.py saved!
   Lines: 621
